# 12. 한국어 챗봇 비교: 생성형 vs 검색형

Kaggle Notebook에서 바로 실행할 수 있는 한국어 챗봇 비교 예제입니다.

- 같은 `chatbotData.csv` 데이터셋 사용
- 생성형 챗봇: `skt/kogpt2-base-v2` + `PreTrainedTokenizerFast`
- 검색형 챗봇: `jhgan/ko-sbert-sts` 질문 임베딩 + 코사인 유사도
- 여러 사용자 질문에 대해 두 방식의 응답을 나란히 비교

> Kaggle에서 모델과 데이터 다운로드가 필요하면 Notebook 오른쪽 설정에서 Internet을 켜 주세요. 생성형 KoGPT2 파인튜닝은 GPU 사용을 권장합니다.

In [ ]:
# Kaggle 환경에서 필요한 패키지를 설치합니다.
# 설치 후 import 오류가 나면 Notebook을 Restart한 뒤 처음부터 다시 실행하세요.
!pip -q install -U transformers sentence-transformers accelerate scikit-learn pandas tqdm

In [ ]:
import os
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from torch.utils.data import Dataset
from transformers import (
    DataCollatorForLanguageModeling,
    GPT2LMHeadModel,
    PreTrainedTokenizerFast,
    Trainer,
    TrainingArguments,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("사용 장치:", device)

## 1. chatbotData.csv 불러오기

Kaggle Dataset으로 `chatbotData.csv`를 추가한 경우 `/kaggle/input` 아래에서 자동으로 찾습니다. 파일이 없으면 공개 GitHub 원본을 `/kaggle/working/chatbotData.csv`로 다운로드합니다.

In [ ]:
def find_chatbot_csv():
    """Kaggle 입력 폴더와 현재 폴더에서 chatbotData.csv를 찾습니다."""
    candidates = []

    search_roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    file_names = ["chatbotData.csv", "ChatbotData.csv", "ChatBotData.csv"]

    for root in search_roots:
        if root.exists():
            for name in file_names:
                candidates.extend(root.glob(f"**/{name}"))

    return candidates[0] if candidates else None


csv_path = find_chatbot_csv()

if csv_path is None:
    # songys/Chatbot_data의 공개 한국어 챗봇 데이터셋입니다.
    download_url = "https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv"
    output_path = Path("/kaggle/working/chatbotData.csv")

    if not Path("/kaggle/working").exists():
        output_path = Path("chatbotData.csv")

    print("chatbotData.csv가 없어 공개 데이터셋을 다운로드합니다.")
    urllib.request.urlretrieve(download_url, output_path)
    csv_path = output_path

print("사용 데이터셋:", csv_path)

chatbot_df = pd.read_csv(csv_path)
chatbot_df.head()

## 2. 데이터 전처리

`Q`는 질문, `A`는 답변입니다. 생성형과 검색형 챗봇이 모두 같은 `Q`, `A` 컬럼을 사용합니다.

In [ ]:
# 데이터셋에 따라 컬럼명이 다를 수 있어 Q/A 컬럼으로 통일합니다.
rename_map = {}

if "Q" not in chatbot_df.columns and "text" in chatbot_df.columns:
    rename_map["text"] = "Q"
if "A" not in chatbot_df.columns and "pair" in chatbot_df.columns:
    rename_map["pair"] = "A"

chatbot_df = chatbot_df.rename(columns=rename_map)

required_columns = {"Q", "A"}
missing_columns = required_columns - set(chatbot_df.columns)
if missing_columns:
    raise ValueError(f"필수 컬럼이 없습니다: {missing_columns}. chatbotData.csv에는 Q, A 컬럼이 필요합니다.")

chatbot_df = chatbot_df[["Q", "A"]].dropna().drop_duplicates().reset_index(drop=True)
chatbot_df["Q"] = chatbot_df["Q"].astype(str).str.strip()
chatbot_df["A"] = chatbot_df["A"].astype(str).str.strip()
chatbot_df = chatbot_df[(chatbot_df["Q"] != "") & (chatbot_df["A"] != "")].reset_index(drop=True)

print("데이터 개수:", len(chatbot_df))
display(chatbot_df.head())

## 3. 생성형 챗봇: KoGPT2 파인튜닝

생성형 챗봇은 질문을 `Q:(질문)\nA` 형식으로 넣고 `model.generate()`로 다음 문장을 생성합니다. 아래 예제는 Kaggle에서 빠르게 실행할 수 있도록 일부 데이터만 사용해 짧게 파인튜닝합니다.

In [ ]:
kogpt2_model_name = "skt/kogpt2-base-v2"

# KoGPT2용 tokenizer입니다. 조건에 맞게 PreTrainedTokenizerFast를 사용합니다.
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    kogpt2_model_name,
    bos_token="</s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    mask_token="<mask>",
)

model = GPT2LMHeadModel.from_pretrained(kogpt2_model_name)
model.resize_token_embeddings(len(tokenizer))
model.to(device)

print("KoGPT2 모델 로딩 완료:", kogpt2_model_name)

In [ ]:
class ChatbotTextDataset(Dataset):
    """Q/A 쌍을 KoGPT2 언어모델 학습용 문장으로 변환합니다."""

    def __init__(self, dataframe, tokenizer, max_length=96):
        self.examples = []
        self.tokenizer = tokenizer
        self.max_length = max_length

        for _, row in dataframe.iterrows():
            text = f"Q:{row['Q']}\nA:{row['A']}{tokenizer.eos_token}"
            encoded = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                padding="max_length",
            )
            self.examples.append(encoded)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        item = {key: torch.tensor(value) for key, value in self.examples[idx].items()}
        item["labels"] = item["input_ids"].clone()
        item["labels"][item["attention_mask"] == 0] = -100
        return item


# Kaggle에서 빠르게 실습할 수 있도록 일부 데이터만 사용합니다.
# 품질을 높이고 싶으면 train_sample_size를 늘리고 num_train_epochs도 늘리세요.
train_sample_size = min(1500, len(chatbot_df))
train_df = chatbot_df.sample(train_sample_size, random_state=42).reset_index(drop=True)

train_dataset = ChatbotTextDataset(train_df, tokenizer, max_length=96)

print("생성형 학습 데이터 개수:", len(train_dataset))

In [ ]:
# KoGPT2를 챗봇 데이터셋 형식에 맞게 짧게 파인튜닝합니다.
# 시간이 부족하면 num_train_epochs를 0으로 두는 대신 이 셀을 건너뛰고 바로 생성 함수를 실행할 수 있습니다.
training_args = TrainingArguments(
    output_dir="/kaggle/working/kogpt2-chatbot" if Path("/kaggle/working").exists() else "kogpt2-chatbot",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=20,
    logging_steps=50,
    save_steps=500,
    save_total_limit=1,
    report_to=[],
    fp16=(device == "cuda"),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()
model.eval()

print("KoGPT2 파인튜닝 완료")

In [ ]:
def generate_chatbot_answer(question, max_new_tokens=50):
    """Q:(질문)\nA 형식 프롬프트를 넣고 KoGPT2가 답변을 생성합니다."""
    prompt = f"Q:{question}\nA"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.92,
            temperature=0.8,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    answer = generated_text.replace(prompt, "", 1).strip()

    # 다음 질문 형식이 이어서 생성되면 답변만 남깁니다.
    for stop_text in ["\nQ:", "\nQ", "Q:"]:
        if stop_text in answer:
            answer = answer.split(stop_text)[0].strip()

    return answer if answer else "답변을 생성하지 못했습니다."

## 4. 검색형 챗봇: Ko-SBERT 임베딩 + 코사인 유사도

검색형 챗봇은 사용자 질문을 임베딩한 뒤, 데이터셋의 `Q` 컬럼 중 가장 비슷한 질문을 찾고 해당 `A`를 반환합니다.

In [ ]:
retrieval_model_name = "jhgan/ko-sbert-sts"
retrieval_model = SentenceTransformer(retrieval_model_name, device=device)

# 전체 질문을 미리 임베딩해 검색 인덱스로 사용합니다.
# normalize_embeddings=True로 정규화하면 코사인 유사도 계산이 안정적입니다.
question_texts = chatbot_df["Q"].tolist()
answer_texts = chatbot_df["A"].tolist()

question_embeddings = retrieval_model.encode(
    question_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("검색형 질문 임베딩 생성 완료:", question_embeddings.shape)

In [ ]:
def retrieve_chatbot_answer(question):
    """사용자 질문과 가장 비슷한 데이터셋 질문을 찾아 해당 답변을 반환합니다."""
    user_embedding = retrieval_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    similarities = cosine_similarity(user_embedding, question_embeddings)[0]
    best_idx = int(np.argmax(similarities))

    return {
        "matched_question": question_texts[best_idx],
        "answer": answer_texts[best_idx],
        "similarity": float(similarities[best_idx]),
    }

## 5. 여러 사용자 질문으로 두 방식 비교하기

In [ ]:
user_questions = [
    "오늘 너무 우울해",
    "친구랑 싸웠어",
    "사랑이 뭘까?",
    "내일 시험인데 걱정돼",
    "심심한데 뭐하지?",
]

comparison_results = []

for question in user_questions:
    generative_answer = generate_chatbot_answer(question)
    retrieval_result = retrieve_chatbot_answer(question)

    comparison_results.append(
        {
            "사용자 질문": question,
            "생성형 응답": generative_answer,
            "검색형 응답": retrieval_result["answer"],
            "검색 매칭 질문": retrieval_result["matched_question"],
            "검색 유사도": round(retrieval_result["similarity"], 4),
        }
    )

comparison_df = pd.DataFrame(comparison_results)
display(comparison_df)

## 6. 두 방식의 차이 정리

| 구분 | 생성형 챗봇 | 검색형 챗봇 |
|---|---|---|
| 핵심 방식 | KoGPT2가 새 문장을 생성 | 가장 비슷한 기존 질문을 찾아 답변 반환 |
| 장점 | 데이터에 없는 표현도 생성 가능 | 답변이 안정적이고 빠름 |
| 단점 | 짧은 학습에서는 어색하거나 반복될 수 있음 | 데이터셋에 없는 상황에는 한계가 있음 |
| 사용 모델 | `skt/kogpt2-base-v2` | `jhgan/ko-sbert-sts` |
| 같은 데이터 사용 방식 | `Q:질문\nA:답변` 문장으로 파인튜닝 | `Q` 임베딩 색인 후 `A` 반환 |